# Colab 8
## Bloom filters
### Data loading 
From the NLTK (Natural Language ToolKit) library, we import a large list of English dictionary words, commonly used by the very first spell-checking programs in Unix-like operating systems.

In [2]:
import nltk
nltk.download('words')

from nltk.corpus import words
word_list = words.words()
print(f'Dictionary length: {len(word_list)}')
print(word_list[:15])

[nltk_data] Downloading package words to
[nltk_data]     /Users/pantelisgiankoulidis/nltk_data...


Dictionary length: 236736
['A', 'a', 'aa', 'aal', 'aalii', 'aam', 'Aani', 'aardvark', 'aardwolf', 'Aaron', 'Aaronic', 'Aaronical', 'Aaronite', 'Aaronitic', 'Aaru']


[nltk_data]   Unzipping corpora/words.zip.


Then we load another dataset from the NLTK Corpora collection: movie_reviews.

The movie reviews are categorized between positive and negative, so we construct a list of words (usually called bag of words) for each category.

In [3]:
from nltk.corpus import movie_reviews
nltk.download('movie_reviews')

neg_reviews = []
pos_reviews = []

for fileid in movie_reviews.fileids('neg'):
  neg_reviews.extend(movie_reviews.words(fileid))
for fileid in movie_reviews.fileids('pos'):
  pos_reviews.extend(movie_reviews.words(fileid))

[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/pantelisgiankoulidis/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


### Task
In this Colab, we will develop a very simplistic spell-checker. By no means you should think of using it for a real-world use case, but it is an interesting exercise to highlight the strenghts and weaknesses of Bloom Filters!

In [5]:
from bloom_filter import BloomFilter

word_filter = BloomFilter(max_elements=236736)

for word in word_list:
  word_filter.add(word)

word_set = set(word_list)

In [6]:
from sys import getsizeof

print(f'Size of word_list (in bytes): {getsizeof(word_list)}')
print(f'Size of word_filter (in bytes): {getsizeof(word_filter)}')
print(f'Size of word_set (in bytes): {getsizeof(word_set)}')

Size of word_list (in bytes): 2055512
Size of word_filter (in bytes): 48
Size of word_set (in bytes): 8388824


In [7]:
%timeit -r 3 "California" in word_list
%timeit -r 3 "California" in word_filter
%timeit -r 3 "California" in word_set

162 μs ± 1.18 μs per loop (mean ± std. dev. of 3 runs, 10,000 loops each)
4.58 μs ± 11.5 ns per loop (mean ± std. dev. of 3 runs, 100,000 loops each)
12.8 ns ± 0.0624 ns per loop (mean ± std. dev. of 3 runs, 100,000,000 loops each)


We now have all the building blocks required to build our spell-checker, and we understand the performance tradeoffs of each datastructure we chose. We are now ready to write a function that takes as arguments (1) a list of words, and (2) any of the 3 dictionary datastructures we constructed. The function returns the number of words which do not appear in the dictionary.

In [19]:
def spell_checker(word_list, dictionary):
    return len([word for word in word_list if word not in dictionary])

wlist = ['hello', 'world', 'this', 'is', 'a', 'test', 'California', 'woojo', 'quen', 'parentt', 'motherr']
print(spell_checker(wlist, word_filter))
print(spell_checker(wlist, word_set))


3
4


We notice that the probabilistic nature of the bloom filter wrongfully identifies the words 'parentt' and 'motherr' as not misspelled, whereas the other misspelled words are correctly identified.